# Notebook 02: Whisper ASR Transcription (base + small)

This notebook runs Whisper CLI to generate `.txt` transcripts.

Outputs:
- `outputs/asr/{ad,cn}/*.txt`  (Whisper **base**)
- `outputs/asr_small/{ad,cn}/*.txt` (Whisper **small**)

> **Prereqs**
- You have `ffmpeg` installed and on PATH.
- Inside your venv: `pip install openai-whisper`

In [ ]:
import subprocess, shlex
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
DATA_AD = PROJECT_ROOT / "data" / "raw" / "ADReSSo21" / "diagnosis" / "train" / "audio" / "ad"
DATA_CN = PROJECT_ROOT / "data" / "raw" / "ADReSSo21" / "diagnosis" / "train" / "audio" / "cn"

OUT_BASE_AD = PROJECT_ROOT / "outputs" / "asr" / "ad"
OUT_BASE_CN = PROJECT_ROOT / "outputs" / "asr" / "cn"
OUT_SMALL_AD = PROJECT_ROOT / "outputs" / "asr_small" / "ad"
OUT_SMALL_CN = PROJECT_ROOT / "outputs" / "asr_small" / "cn"

for p in [OUT_BASE_AD, OUT_BASE_CN, OUT_SMALL_AD, OUT_SMALL_CN]:
    p.mkdir(parents=True, exist_ok=True)

def run_whisper(model: str, in_dir: Path, out_dir: Path, language="en"):
    wavs = sorted(in_dir.glob("*.wav"))
    print(f"Running Whisper model={model} on {len(wavs)} files from {in_dir.name} → {out_dir}")
    for i, wav in enumerate(wavs, 1):
        cmd = f'whisper "{wav}" --model {model} --language {language} --task transcribe --output_format txt --output_dir "{out_dir}" --verbose False'
        subprocess.run(cmd, shell=True, check=True)
        if i % 10 == 0:
            print(f"  done {i}/{len(wavs)}")

# === Quick smoke test on 3 files (recommended) ===
test_ad = sorted(DATA_AD.glob("*.wav"))[:3]
print("Smoke test files:", [p.name for p in test_ad])

# Run smoke test (base) to outputs/asr_smoke
SMOKE = PROJECT_ROOT / "outputs" / "asr_smoke"
SMOKE.mkdir(parents=True, exist_ok=True)
for wav in test_ad:
    cmd = f'whisper "{wav}" --model base --language en --task transcribe --output_format txt --output_dir "{SMOKE}" --verbose False'
    subprocess.run(cmd, shell=True, check=True)

print("Smoke transcripts:", [p.name for p in sorted(SMOKE.glob("*.txt"))])

## Full transcription

Uncomment and run the following cells **one block at a time**.

In [ ]:
# --- Full run: Whisper base ---
# run_whisper("base", DATA_AD, OUT_BASE_AD)
# run_whisper("base", DATA_CN, OUT_BASE_CN)

In [ ]:
# --- Full run: Whisper small ---
# run_whisper("small", DATA_AD, OUT_SMALL_AD)
# run_whisper("small", DATA_CN, OUT_SMALL_CN)

## Verify counts

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
base_ad = list((PROJECT_ROOT/"outputs"/"asr"/"ad").glob("*.txt"))
base_cn = list((PROJECT_ROOT/"outputs"/"asr"/"cn").glob("*.txt"))
small_ad = list((PROJECT_ROOT/"outputs"/"asr_small"/"ad").glob("*.txt"))
small_cn = list((PROJECT_ROOT/"outputs"/"asr_small"/"cn").glob("*.txt"))

print("base AD:", len(base_ad), "base CN:", len(base_cn))
print("small AD:", len(small_ad), "small CN:", len(small_cn))